In [1]:
import moviepy
from moviepy import *
from moviepy.video import *
'''
# loading video gfg
clip = VideoFileClip("../Data/modelUnderwaters.mp4")


# getting subclip from it
clip = clip.subclipped(0, 4)

# applying speed effect
effect = vfx.AccelDecel(1, abruptness=0, soonness=1)

clip = effect.apply(clip)



# showing final clip
#clip.display_in_notebook()


#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS

length = 10

clip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)
clip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)


combined = clips_array([[clip1], [clip2]])

effect = vfx.Margin(left=960, right=960)
combined = effect.apply(combined)
effect = vfx.Resize((1920,1080) )
combined = effect.apply(combined)

#combined = combined.resize(height=1920)


effect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)
#clip = effect.apply(clip)
#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)

#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))
combined.display_in_notebook
#combined.write_videofile('test.mp4')

combined.write_videofile("../outputs/result.mp4")

'''

'\n# loading video gfg\nclip = VideoFileClip("../Data/modelUnderwaters.mp4")\n\n\n# getting subclip from it\nclip = clip.subclipped(0, 4)\n\n# applying speed effect\neffect = vfx.AccelDecel(1, abruptness=0, soonness=1)\n\nclip = effect.apply(clip)\n\n\n\n# showing final clip\n#clip.display_in_notebook()\n\n\n#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS\n\nlength = 10\n\nclip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)\nclip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)\n\n\ncombined = clips_array([[clip1], [clip2]])\n\neffect = vfx.Margin(left=960, right=960)\ncombined = effect.apply(combined)\neffect = vfx.Resize((1920,1080) )\ncombined = effect.apply(combined)\n\n#combined = combined.resize(height=1920)\n\n\neffect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n#clip = effect.apply(clip)\n#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n\n#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))\ncombin

In [2]:
import accelerate

print(accelerate.__version__)

1.12.0


c:\Users\danre\Documents\GitHub\SwimVisionProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


import torch
import numpy as np
import cv2
import matplotlib
import argparse
import os
import time
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
index = 0
OUT_DIR = '../outputs'
os.makedirs(OUT_DIR, exist_ok=True)


fileName = 'meSwim.mp4'
inputPath = '../Data/' + fileName
cap = cv2.VideoCapture(inputPath)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
video_fps = int(cap.get(5)) 

from pathlib import Path

# my_file = Path("/path/to/file")
# if my_file.is_file():
#     # file exists

while True:
    save_name = fileName + str(index)
    outputPath = Path(f'{OUT_DIR}/{save_name}.mp4')
    if outputPath.is_file():
        index += 1
        continue
    else:
        out = cv2.VideoWriter(
            f'{OUT_DIR}/{save_name}.mp4', 
            cv2.VideoWriter_fourcc(*'mp4v'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        break
model_name = 'usyd-community/vitpose-plus-huge'



save_name = 'Ivan'#args.input.split(os.path.sep)[-1].split('.')[0]
# Define codec and create VideoWriter object.

# Load detector.
person_image_processor = AutoProcessor.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365'
)
person_model = RTDetrForObjectDetection.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365', device_map=device
)
# Load ViTPose.
#print(f"Pose Model: {'../models/VitPose-s_RePoGen.pth'}")
image_processor = AutoProcessor.from_pretrained(model_name)
model = VitPoseForPoseEstimation.from_pretrained(pretrained_model_name_or_path=model_name, device_map=device)


#This is sadly chatted :(
# modelCustom = torch.load('../Models/VitPose-s_RePoGen.pth', device, weights_only=False)
# model.load_state_dict(modelCustom)
# image_processor.load_state_dict(modelCustom)
# image_processor.eval()
# model.eval()

True


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
edges = [
    (0, 1), (0, 2), (2, 4), (1, 3), (6, 8), (8, 10),
    (5, 7), (7, 9), (5, 11), (11, 13), (13, 15), (6, 12),
    (12, 14), (14, 16), (5, 6), (11, 12)
]
def detect_objects(image):
    """
    :param image: Image in PIL image format.
    Returns:
        person_boxes: Bboxes of persons in [x, y, w, h] format.
    """
    det_time_start = time.time()
    inputs = person_image_processor(
        images=image, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        outputs = person_model(**inputs)
    target_sizes = torch.tensor([(image.height, image.width)])
    
    results = person_image_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=0.3
    )
    det_time_end = time.time()
    det_fps = 1 / (det_time_end-det_time_start)
    
    # Extract the first result, as we can pass multiple images at a time.
    result = results[0]
    
    # In COCO dataset, humans labels have index 0.
    person_boxes_xyxy = result['boxes'][result['labels'] == 0]
    person_boxes_xyxy = person_boxes_xyxy.cpu().numpy()
    
    # Convert boxes from (x1, y1, x2, y2) to (x1, y1, w, h) format.
    person_boxes = person_boxes_xyxy.copy()
    person_boxes[:, 2] = person_boxes[:, 2] - person_boxes[:, 0]
    person_boxes[:, 3] = person_boxes[:, 3] - person_boxes[:, 1]
    return person_boxes, det_fps
def detect_pose(image, person_boxes):
    """
    :param image: Image in PIL image format.
    :param person_bboxes: Batched person boxes in [[x, y, w, h], ...] format.
    """
    pose_time_start = time.time()
    inputs = image_processor(
        image, boxes=[person_boxes], return_tensors='pt'
    ).to(device)
    
    dataset_index = torch.tensor([0], device=device) # must be a tensor of shape (batch_size,)
    if len(person_boxes) != 0:
        if 'plus' in model_name:
            with torch.no_grad():
                outputs = model(**inputs, dataset_index=dataset_index)
        else:
            with torch.no_grad():
                outputs = model(**inputs)
        
        pose_results = image_processor.post_process_pose_estimation(
            outputs, boxes=[person_boxes]
        )
    pose_time_end = time.time()
    pose_fps = 1 / (pose_time_end-pose_time_start)
    if len(person_boxes) == 0:
        return [], pose_fps
    image_pose_result = pose_results[0]
    
    return image_pose_result, pose_fps
def draw_keypoints(outputs, image):
    """
    :param outputs: Outputs from the keypoint detector.
    :param image: Image in PIL Image format.
    Returns:
        image: Annotated image Numpy array format.
    """
    image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    # the `outputs` is list which in-turn contains the dictionaries 
    for i, pose_result in enumerate(outputs):
        keypoints = pose_result['keypoints'].cpu().detach().numpy()
        # proceed to draw the lines if the confidence score is above 0.9
        keypoints = keypoints[:, :].reshape(-1, 2)
        if keypoints.shape[0] != 0 and keypoints.shape[0] != 17:
            print(keypoints.shape[0]) 
        for p in range(keypoints.shape[0]):
            # draw the keypoints
            cv2.circle(image, (int(keypoints[p, 0]), int(keypoints[p, 1])), 
                        3, (0, 0, 255), thickness=-1, lineType=cv2.FILLED)
            # uncomment the following lines if you want to put keypoint number
            cv2.putText(image, f'{p}', (int(keypoints[p, 0]+10), int(keypoints[p, 1]-5)),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        for ie, e in enumerate(edges):
            # get different colors for the edges
            rgb = matplotlib.colors.hsv_to_rgb([
                ie/float(len(edges)), 1.0, 1.0
            ])
            rgb = rgb*255
            # join the keypoint pairs to draw the skeletal structure
            cv2.line(image, (int(keypoints[e, 0][0]), int(keypoints[e, 1][0])),
                    (int(keypoints[e, 0][1]), int(keypoints[e, 1][1])),
                    tuple(rgb), 2, lineType=cv2.LINE_AA)
    return image

body_edges = [(6, 8), (8, 10),
    (5, 7), (7, 9), (5, 11), (11, 13), (13, 15), (6, 12),
    (12, 14), (14, 16), (5, 6), (11, 12)]

def getKeypoints(outputs):
    out = []
    if outputs:
        keypoints = outputs[0]['keypoints'].cpu().detach().numpy()
        keypoints = keypoints[:, :].reshape(-1, 2)
        for p in range(keypoints.shape[0]):
            out.append([int(keypoints[p, 0]), int(keypoints[p, 1])])
    return out

def drawSlopes(image, slopes, points):
    if slopes and points:
        for i, edge in enumerate(body_edges):
            average = getAverageOffsetCoord(points[edge[0]], points[edge[1]])
            cv2.putText(image, f'{slopes[i]}', (average[0], average[1]),
                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)           
    return image



def getSlopes(outputs, image, points):
    slopes = []
    if outputs and points:
        for i, edge in enumerate(body_edges):
            slopes.append(slope(points[edge[0]], points[edge[1]]))
    return slopes


def switchOpenCVNormal(image, newkps):
    width, height = image.size
    for i in range(len(newkps)):
        newkps[i][1] = height - newkps[i][1]
    return newkps

def getAverageOffsetCoord(coord1, coord2):
    return [int((coord1[0] + coord2[0])/2 -10), int((coord1[1] + coord2[1])/2)]

def slope(coord1, coord2):
    if coord1[0] - coord2[0] != 0:
        return ((coord1[1] - coord2[1]) / (coord1[0] - coord2[0])) 
    else:
        return 100


In [5]:
frame_count = 0 # To count total frames.
total_fps = 0 # To get the final frames per second.
masterSlopeList = []
while cap.isOpened():
    ret, frame = cap.read()
    if ret:
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        image = Image.fromarray(frame_rgb)
        start_time = time.time()
        bboxes, det_fps = detect_objects(image=image)
        image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
        result = draw_keypoints(image_pose_result, image)
        kps = getKeypoints(image_pose_result)
        normalkps = switchOpenCVNormal(image, kps)
        listSlopes = getSlopes(image_pose_result, image, normalkps)
        result = drawSlopes(result, listSlopes, kps)
        masterSlopeList.append(listSlopes)
        end_time = time.time()
        forward_pass_time = end_time - start_time
            
        # Get the current fps.
        fps = 1 / (forward_pass_time)
        # Add `fps` to `total_fps`.
        total_fps += fps
        # Increment frame count.
        frame_count += 1
        cv2.putText(
            result,
            f"FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
            (15, 25),
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=1.0,
            color=(0, 0, 255),
            thickness=2,
            lineType=cv2.LINE_AA,
        )
        out.write(result)
        cv2.imshow('Prediction', result)
        # Press `q` to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    else:
        break
 # Release VideoCapture().
cap.release()

# Close all frames and video windows.
cv2.destroyAllWindows()
out.release()
avg_fps = total_fps/frame_count
print(f"Average FPS: {avg_fps}")

Average FPS: 6.981245471356228


In [6]:
print(masterSlopeList)
print(len(masterSlopeList))
print(frame_count)

[[0.3467741935483871, 0.15, 0.31666666666666665, 0.19626168224299065, -0.20284697508896798, -0.1286549707602339, -0.28402366863905326, -0.1978021978021978, -0.14634146341463414, -0.22297297297297297, 0.75, 100], [0.26956521739130435, 0.2692307692307692, 0.37593984962406013, 0.3333333333333333, -0.2037037037037037, -0.16666666666666666, -0.22033898305084745, -0.19494584837545126, -0.08641975308641975, -0.11278195488721804, 1.0, 1.375], [0.46464646464646464, 0.23423423423423423, 0.5185185185185185, 0.3739130434782609, -0.18411552346570398, -0.19428571428571428, 0.0603448275862069, -0.18725099601593626, -0.12790697674418605, 0.1810344827586207, -8.0, 0.5], [0.5, 0.4895833333333333, 0.4657534246575342, 0.6712328767123288, -0.2076923076923077, -0.18407960199004975, 0.13008130081300814, -0.19424460431654678, -0.2768361581920904, 0.2517482517482518, 0.813953488372093, 1.4], [0.6219512195121951, 0.7283950617283951, -1.0625, 0.41818181818181815, -0.19771863117870722, -0.18181818181818182, 0.103

In [7]:

'''
def findFirstFrame(slopeList):
    for slopeSet in slopeList:
        curSlope = 

'''

'\ndef findFirstFrame(slopeList):\n    for slopeSet in slopeList:\n        curSlope = \n\n'